In [7]:
# *********************FINAL 2********************
import json
import html
import webbrowser
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler

# ============================================================
# Configuration
# ============================================================

INPUT_CSV = "synthetic_step1_biostat_dataset.csv"
OUTPUT_HTML = "high-dimensional_3d_fullpage_with_rotatable_cluster_selector.html"
ROTATION_DELTA = 0.01
TOP_K_FEATURES = 5

# Main 3D orb marker sizes
DEFAULT_MARKER_SIZE = 7
SELECTED_MARKER_SIZE = 10
NONSELECTED_MARKER_SIZE = 5

# Non-selected points in the 3D orb fade toward near-black
NEAR_BLACK_COLOR = "rgba(8,8,8,0.10)"

# ============================================================
# Load data
# ============================================================

df = pd.read_csv(INPUT_CSV)

required_cols = {"student_id", "step1_outcome"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

student_ids = df["student_id"].astype(str).copy()
y = df["step1_outcome"].copy()

raw_features = df.drop(columns=["student_id", "step1_outcome"]).copy()
df_model = raw_features.copy()

categorical_cols = df_model.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in df_model.columns if c not in categorical_cols]

if len(categorical_cols) > 0:
    encoder = OrdinalEncoder()
    df_model[categorical_cols] = encoder.fit_transform(df_model[categorical_cols])

if y.dtype == "object" or str(y.dtype).startswith("category"):
    y_encoder = LabelEncoder()
    _ = y_encoder.fit_transform(y.astype(str))
    y_plot = y.astype(str)
else:
    y_plot = y.astype(str)

# ============================================================
# Standardize and build similarity representation
# ============================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model)

K = rbf_kernel(X_scaled, gamma=0.5)

D = 1 - K
D = (D + D.T) / 2
np.fill_diagonal(D, 0.0)

coords = TSNE(
    n_components=3,
    metric="precomputed",
    init="random",
    random_state=42,
    perplexity=30
).fit_transform(D)

plot_df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "z": coords[:, 2],
    "student_id": student_ids,
    "step1_outcome": y_plot
})

# ============================================================
# Group naming helpers
# ============================================================

unique_labels = list(pd.Series(y_plot).astype(str).unique())
if len(unique_labels) < 2:
    raise ValueError("The target column 'step1_outcome' must contain at least two classes.")

def choose_pass_fail_labels(labels):
    lower_map = {label: label.lower() for label in labels}
    pass_candidates = [label for label in labels if "pass" in lower_map[label]]
    fail_candidates = [label for label in labels if "fail" in lower_map[label]]

    if pass_candidates and fail_candidates:
        return pass_candidates[0], fail_candidates[0]

    counts = pd.Series(y_plot).value_counts()
    ordered = counts.index.tolist()
    return ordered[0], ordered[1]

pass_label, fail_label = choose_pass_fail_labels(unique_labels)

mask_pass = (y_plot.astype(str) == str(pass_label)).to_numpy()
mask_fail = (y_plot.astype(str) == str(fail_label)).to_numpy()

if mask_pass.sum() == 0 or mask_fail.sum() == 0:
    raise ValueError("Could not identify non-empty pass/fail groups.")

pass_centroid = X_scaled[mask_pass].mean(axis=0)
fail_centroid = X_scaled[mask_fail].mean(axis=0)

# ============================================================
# Per-student profile summaries
# ============================================================

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return None

def format_value(v):
    if pd.isna(v):
        return "missing"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.3f}"
    return str(v)

def html_list(items):
    if not items:
        return "<li>None identified</li>"
    return "".join(f"<li>{html.escape(item)}</li>" for item in items)

numeric_stats = {}
for col in numeric_cols:
    col_values = pd.to_numeric(raw_features[col], errors="coerce")
    sigma = float(np.nanstd(col_values.to_numpy(), ddof=0))
    if not np.isfinite(sigma) or sigma < 1e-12:
        sigma = 1.0

    pass_mean = float(np.nanmean(pd.to_numeric(raw_features.loc[mask_pass, col], errors="coerce")))
    fail_mean = float(np.nanmean(pd.to_numeric(raw_features.loc[mask_fail, col], errors="coerce")))

    numeric_stats[col] = {
        "std": sigma,
        "pass_mean": pass_mean,
        "fail_mean": fail_mean
    }

categorical_stats = {}
for col in categorical_cols:
    pass_mode_series = raw_features.loc[mask_pass, col].mode(dropna=True)
    fail_mode_series = raw_features.loc[mask_fail, col].mode(dropna=True)

    pass_mode = None if pass_mode_series.empty else pass_mode_series.iloc[0]
    fail_mode = None if fail_mode_series.empty else fail_mode_series.iloc[0]

    categorical_stats[col] = {
        "pass_mode": pass_mode,
        "fail_mode": fail_mode
    }

student_profiles = []

for i in range(len(df)):
    row_raw = raw_features.iloc[i]
    row_scaled = X_scaled[i]

    dist_pass = float(np.linalg.norm(row_scaled - pass_centroid))
    dist_fail = float(np.linalg.norm(row_scaled - fail_centroid))
    overall_similarity = pass_label if dist_pass < dist_fail else fail_label

    pass_like_scores = []
    fail_like_scores = []
    difference_signals = []

    for col in numeric_cols:
        val = safe_float(row_raw[col])
        if val is None or pd.isna(val):
            continue

        stats = numeric_stats[col]
        std = stats["std"]
        d_pass = abs(val - stats["pass_mean"]) / std
        d_fail = abs(val - stats["fail_mean"]) / std
        separation = abs(d_fail - d_pass)

        if d_pass < d_fail:
            pass_like_scores.append((
                col,
                separation,
                f"{col}: {val:.3f} is closer to the {pass_label} mean "
                f"({stats['pass_mean']:.3f}) than the {fail_label} mean ({stats['fail_mean']:.3f})"
            ))
        elif d_fail < d_pass:
            fail_like_scores.append((
                col,
                separation,
                f"{col}: {val:.3f} is closer to the {fail_label} mean "
                f"({stats['fail_mean']:.3f}) than the {pass_label} mean ({stats['pass_mean']:.3f})"
            ))

        midpoint = 0.5 * (stats["pass_mean"] + stats["fail_mean"])
        direction = (
            f"leans toward {pass_label}" if abs(val - stats["pass_mean"]) < abs(val - stats["fail_mean"])
            else f"leans toward {fail_label}"
        )
        difference_signals.append((
            col,
            separation,
            f"{col}: {val:.3f}; class midpoint {midpoint:.3f}; this value {direction}"
        ))

    for col in categorical_cols:
        val = row_raw[col]
        stats = categorical_stats[col]
        pass_mode = stats["pass_mode"]
        fail_mode = stats["fail_mode"]

        matched_pass = (pd.notna(val) and pd.notna(pass_mode) and val == pass_mode)
        matched_fail = (pd.notna(val) and pd.notna(fail_mode) and val == fail_mode)

        if matched_pass and not matched_fail:
            pass_like_scores.append((
                col,
                1.0,
                f"{col}: '{val}' matches the most common {pass_label} value"
            ))
        elif matched_fail and not matched_pass:
            fail_like_scores.append((
                col,
                1.0,
                f"{col}: '{val}' matches the most common {fail_label} value"
            ))

        if pass_mode != fail_mode and pd.notna(val):
            leaning = None
            if matched_pass and not matched_fail:
                leaning = pass_label
            elif matched_fail and not matched_pass:
                leaning = fail_label

            if leaning is not None:
                difference_signals.append((
                    col,
                    1.0,
                    f"{col}: '{val}' aligns with the most common {leaning} category"
                ))

    pass_like_scores = sorted(pass_like_scores, key=lambda t: t[1], reverse=True)[:TOP_K_FEATURES]
    fail_like_scores = sorted(fail_like_scores, key=lambda t: t[1], reverse=True)[:TOP_K_FEATURES]
    difference_signals = sorted(difference_signals, key=lambda t: t[1], reverse=True)[:TOP_K_FEATURES]

    raw_characteristics = [f"{col}: {format_value(row_raw[col])}" for col in raw_features.columns]

    coords_i = coords[i]
    profile_html = f"""
    <div class="profile-card">
        <div class="profile-title">Student Profile</div>
        <div><b>Student ID:</b> {html.escape(str(student_ids.iloc[i]))}</div>
        <div><b>Observed Outcome:</b> {html.escape(str(y_plot.iloc[i]))}</div>
        <div><b>Embedding Coordinates:</b> ({coords_i[0]:.3f}, {coords_i[1]:.3f}, {coords_i[2]:.3f})</div>
        <div><b>Nearest Group Profile:</b> {html.escape(str(overall_similarity))}</div>
        <div><b>Distance to {html.escape(str(pass_label))} centroid:</b> {dist_pass:.3f}</div>
        <div><b>Distance to {html.escape(str(fail_label))} centroid:</b> {dist_fail:.3f}</div>

        <hr class="panel-rule">

        <div class="section-header">Most similar to {html.escape(str(pass_label))}</div>
        <ul>{html_list([item[2] for item in pass_like_scores])}</ul>

        <div class="section-header">Most similar to {html.escape(str(fail_label))}</div>
        <ul>{html_list([item[2] for item in fail_like_scores])}</ul>

        <div class="section-header">Strongest differentiating characteristics</div>
        <ul>{html_list([item[2] for item in difference_signals])}</ul>

        <div class="section-header">Raw characteristics</div>
        <ul>{html_list(raw_characteristics)}</ul>
    </div>
    """
    student_profiles.append(profile_html)

# ============================================================
# JSON payloads for browser-side cluster summaries
# ============================================================

raw_feature_records = raw_features.copy()
for col in raw_feature_records.columns:
    raw_feature_records[col] = raw_feature_records[col].where(pd.notna(raw_feature_records[col]), None)

# Keep pass/fail explicitly green/red when available
outcome_color_map = {}
other_palette = [
    "#60a5fa", "#fbbf24", "#a78bfa", "#f472b6", "#22d3ee", "#fb7185"
]

if pass_label in unique_labels:
    outcome_color_map[str(pass_label)] = "#22c55e"
if fail_label in unique_labels:
    outcome_color_map[str(fail_label)] = "#ef4444"

palette_idx = 0
for label in unique_labels:
    s = str(label)
    if s not in outcome_color_map:
        outcome_color_map[s] = other_palette[palette_idx % len(other_palette)]
        palette_idx += 1

client_payload = {
    "student_ids": student_ids.astype(str).tolist(),
    "outcomes": y_plot.astype(str).tolist(),
    "coords": coords.tolist(),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "raw_records": raw_feature_records.to_dict(orient="records"),
    "x_scaled": X_scaled.tolist(),
    "pass_centroid": pass_centroid.tolist(),
    "fail_centroid": fail_centroid.tolist(),
    "pass_label": str(pass_label),
    "fail_label": str(fail_label),
    "top_k": TOP_K_FEATURES,
    "base_colors": [outcome_color_map[str(v)] for v in y_plot.astype(str).tolist()],
    "near_black_color": NEAR_BLACK_COLOR
}

# ============================================================
# Plotly figure
# ============================================================

base_colors = [outcome_color_map[str(v)] for v in plot_df["step1_outcome"].astype(str).tolist()]

customdata = np.column_stack([
    np.arange(len(plot_df)),
    plot_df["student_id"].astype(str).to_numpy(),
    plot_df["step1_outcome"].astype(str).to_numpy(),
    plot_df["x"].to_numpy(),
    plot_df["y"].to_numpy(),
    plot_df["z"].to_numpy(),
])

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=plot_df["x"],
    y=plot_df["y"],
    z=plot_df["z"],
    mode="markers",
    showlegend=False,
    customdata=customdata,
    marker=dict(
        size=DEFAULT_MARKER_SIZE,
        color=base_colors,
        opacity=1.0,
        line=dict(width=0)
    ),
    hovertemplate=(
        "<b>Student ID:</b> %{customdata[1]}<br>"
        "<b>Outcome:</b> %{customdata[2]}<br>"
        "<b>x:</b> %{customdata[3]:.3f}<br>"
        "<b>y:</b> %{customdata[4]:.3f}<br>"
        "<b>z:</b> %{customdata[5]:.3f}"
        "<extra></extra>"
    )
))

for label in unique_labels:
    fig.add_trace(go.Scatter3d(
        x=[None],
        y=[None],
        z=[None],
        mode="markers",
        name=str(label),
        marker=dict(size=8, color=outcome_color_map[str(label)]),
        showlegend=True
    ))

fig.update_layout(
    title="Hilbert-Space Similarity Map of Step 1 Student Profiles",
    paper_bgcolor="#000000",
    plot_bgcolor="#000000",
    scene=dict(
        domain=dict(x=[0.0, 1.0], y=[0.0, 1.0]),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        camera=dict(
            eye=dict(x=1.8, y=1.8, z=0.9),
            up=dict(x=0, y=0, z=1)
        )
    ),
    margin=dict(l=0, r=0, t=0, b=0),
    dragmode="turntable",
    legend=dict(
        bgcolor="rgba(0,0,0,0.35)",
        x=0.02,
        y=0.98,
        font=dict(color="white")
    )
)

plot_html = pio.to_html(
    fig,
    include_plotlyjs=True,
    full_html=False,
    config={
        "scrollZoom": True,
        "displayModeBar": True,
        "responsive": True
    }
)

profiles_json = json.dumps(student_profiles)
client_json = json.dumps(client_payload)

page_html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="utf-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1" />
    <title>Hilbert-Space Similarity Map of Step 1 Student Profiles</title>
    <style>
        html, body {{
            margin: 0;
            padding: 0;
            width: 100%;
            height: 100%;
            overflow: hidden;
            background: #000;
            color: #fff;
            font-family: Arial, Helvetica, sans-serif;
        }}

        body {{
            position: relative;
        }}

        #plot-fullscreen {{
            position: fixed;
            inset: 0;
            width: 100vw;
            height: 100vh;
            background: #000;
            z-index: 1;
        }}

        #plot-wrap {{
            position: absolute;
            inset: 0;
            width: 100%;
            height: 100%;
        }}

        #plot-wrap > div,
        #plot-wrap .plotly,
        #plot-wrap .js-plotly-plot,
        #plot-wrap .plot-container {{
            width: 100% !important;
            height: 100% !important;
        }}

        #top-header {{
            position: fixed;
            top: 12px;
            left: 12px;
            right: 92px;
            z-index: 20;
            max-width: min(1200px, calc(100vw - 120px));
            background: rgba(10, 10, 10, 0.72);
            border: 1px solid rgba(255,255,255,0.12);
            border-radius: 14px;
            padding: 14px 16px;
            backdrop-filter: blur(8px);
            box-shadow: 0 8px 28px rgba(0,0,0,0.35);
        }}

        #top-header h1 {{
            margin: 0 0 8px 0;
            font-size: 24px;
            line-height: 1.2;
        }}

        #top-header p {{
            margin: 6px 0;
            color: #e0e0e0;
            font-size: 14px;
            line-height: 1.5;
        }}

        #side-panel {{
            position: fixed;
            top: 150px;
            right: 12px;
            bottom: 12px;
            width: 410px;
            max-width: calc(100vw - 24px);
            overflow-y: auto;
            z-index: 20;
            background: rgba(10, 10, 10, 0.78);
            border: 1px solid rgba(255,255,255,0.12);
            border-radius: 14px;
            padding: 16px;
            backdrop-filter: blur(10px);
            box-shadow: 0 8px 28px rgba(0,0,0,0.38);
        }}

        #side-panel h2 {{
            margin: 0 0 10px 0;
            font-size: 18px;
        }}

        #side-panel p {{
            margin: 0 0 10px 0;
            color: #d8d8d8;
            line-height: 1.45;
            font-size: 14px;
        }}

        .hint-box {{
            background: rgba(255,255,255,0.04);
            border: 1px solid rgba(255,255,255,0.10);
            border-radius: 10px;
            padding: 12px;
            margin-top: 12px;
        }}

        .profile-card {{
            font-size: 14px;
            line-height: 1.5;
        }}

        .profile-title {{
            font-size: 17px;
            font-weight: 700;
            margin-bottom: 10px;
        }}

        .section-header {{
            margin-top: 14px;
            margin-bottom: 6px;
            font-weight: 700;
            color: #ffffff;
        }}

        .panel-rule {{
            border: none;
            border-top: 1px solid rgba(255,255,255,0.12);
            margin: 14px 0;
        }}

        ul {{
            margin-top: 6px;
            margin-bottom: 10px;
            padding-left: 20px;
        }}

        li {{
            margin-bottom: 6px;
            color: #d9d9d9;
        }}

        #rotate-controls {{
            position: fixed;
            top: 58px;
            right: 12px;
            z-index: 30;
        }}

        #rotate-toggle {{
            background: rgba(20, 20, 20, 0.88);
            color: white;
            border: 1px solid rgba(255,255,255,0.25);
            border-radius: 8px;
            padding: 8px 12px;
            font-size: 14px;
            cursor: pointer;
            box-shadow: 0 2px 10px rgba(0,0,0,0.35);
        }}

        #rotate-toggle:hover {{
            background: rgba(40, 40, 40, 0.96);
        }}

        #selection-panel {{
            position: fixed;
            left: 12px;
            bottom: 12px;
            width: 380px;
            z-index: 25;
            background: rgba(10, 10, 10, 0.78);
            border: 1px solid rgba(255,255,255,0.12);
            border-radius: 14px;
            padding: 12px;
            backdrop-filter: blur(10px);
            box-shadow: 0 8px 28px rgba(0,0,0,0.38);
        }}

        #selection-panel h3 {{
            margin: 0 0 8px 0;
            font-size: 16px;
        }}

        #selection-panel p {{
            margin: 6px 0 8px 0;
            color: #d8d8d8;
            font-size: 13px;
            line-height: 1.4;
        }}

        #selector-controls {{
            display: flex;
            gap: 8px;
            flex-wrap: wrap;
            margin-bottom: 10px;
        }}

        .sel-btn {{
            background: rgba(20, 20, 20, 0.88);
            color: white;
            border: 1px solid rgba(255,255,255,0.20);
            border-radius: 8px;
            padding: 7px 10px;
            font-size: 13px;
            cursor: pointer;
        }}

        .sel-btn.active {{
            border-color: rgba(255,255,255,0.9);
            background: rgba(55, 55, 55, 0.95);
        }}

        #selector-wrap {{
            position: relative;
            width: 100%;
            height: 250px;
            border-radius: 10px;
            overflow: hidden;
            background: rgba(255,255,255,0.03);
            border: 1px solid rgba(255,255,255,0.12);
        }}

        #selector-svg {{
            width: 100%;
            height: 100%;
            display: block;
            touch-action: none;
            cursor: crosshair;
        }}

        .selector-caption {{
            margin-top: 8px;
            font-size: 12px;
            color: #cfcfcf;
        }}

        .split-block {{
            margin-top: 14px;
            padding-top: 12px;
            border-top: 1px solid rgba(255,255,255,0.10);
        }}

        .metric-grid {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 8px;
            margin-top: 10px;
        }}

        .metric-card {{
            background: rgba(255,255,255,0.04);
            border: 1px solid rgba(255,255,255,0.08);
            border-radius: 8px;
            padding: 8px;
        }}

        .metric-card .label {{
            font-size: 12px;
            color: #cfcfcf;
        }}

        .metric-card .value {{
            font-size: 16px;
            font-weight: 700;
            margin-top: 4px;
        }}

        .small-status {{
            margin-top: 8px;
            font-size: 12px;
            color: #d5d5d5;
            opacity: 0.95;
        }}

        @media (max-width: 1200px) {{
            #side-panel {{
                width: 360px;
            }}
        }}

        @media (max-width: 1100px) {{
            #top-header {{
                max-width: calc(100vw - 24px);
                right: 12px;
            }}

            #side-panel {{
                top: auto;
                left: 12px;
                right: 12px;
                bottom: 290px;
                width: auto;
                height: 34vh;
                max-width: none;
            }}

            #selection-panel {{
                width: calc(100vw - 24px);
            }}
        }}
    </style>
</head>
<body>
    <div id="plot-fullscreen">
        <div id="plot-wrap">
            {plot_html}
        </div>
    </div>

    <div id="top-header">
        <h1>Hilbert-Space Similarity Map of Step 1 Student Profiles</h1>
        <p>
            This view shows the full 3D similarity structure across students. Each point is a student, and nearby points represent students
            with more similar multivariate predictor profiles after standardization and kernel-based similarity mapping.
        </p>
        <p>
            Use the orb to inspect clusters, outliers, and overlap between pass/fail patterns. Hover over a point to open an individual profile.
            The cluster focus selector now supports a separate manual view rotation so you can rotate that selector to find a better angle before drawing or moving the selection.
        </p>
    </div>

    <div id="rotate-controls">
        <button id="rotate-toggle">Pause Rotation</button>
    </div>

    <div id="selection-panel">
        <h3>Cluster Focus Selector</h3>
        <p>
            Use <b>Rotate View</b> to manually rotate the selector projection only. Use <b>Draw / Move Shape</b> to create or slide the circle/ellipse.
            The main 3D orb remains unchanged and keeps its own rotation behavior.
        </p>

        <div id="selector-controls">
            <button class="sel-btn active" id="tool-draw">Draw / Move Shape</button>
            <button class="sel-btn" id="tool-rotate">Rotate View</button>
            <button class="sel-btn active" id="mode-ellipse">Ellipse</button>
            <button class="sel-btn" id="mode-circle">Circle</button>
            <button class="sel-btn" id="clear-selection">Clear Selection</button>
            <button class="sel-btn" id="reset-selector-view">Reset View</button>
        </div>

        <div id="selector-wrap">
            <svg id="selector-svg" viewBox="0 0 380 250" preserveAspectRatio="none"></svg>
        </div>

        <div class="selector-caption">
            Rotate mode changes only the selector projection. Draw mode lets you create a shape or drag an existing one.
        </div>
        <div class="small-status" id="selector-view-status"></div>
    </div>

    <div id="side-panel">
        <h2>Hover and Cluster Explorer</h2>
        <p>
            Hover over a point to display the student's characteristics. Draw a cluster in the selector to summarize the students inside the selected region and compare them with the
            <b>{html.escape(str(pass_label))}</b> and <b>{html.escape(str(fail_label))}</b> groups.
        </p>

        <div class="hint-box" id="profile-content">
            <div class="profile-title">No student selected yet</div>
            <p>
                Move your cursor over a point in the 3D map. This panel will update with the student's ID, outcome,
                embedding coordinates, raw features, and the strongest pass/fail similarity signals.
            </p>
        </div>

        <div class="hint-box split-block" id="cluster-content">
            <div class="profile-title">No cluster selected yet</div>
            <p>
                Draw a circle or ellipse in the lower-left selector. This panel will summarize the selected cluster,
                its pass/fail composition, its centroid in embedding space, and its strongest distinguishing characteristics.
            </p>
        </div>
    </div>

    <script>
        const STUDENT_PROFILES = {profiles_json};
        const CLIENT_DATA = {client_json};
    </script>

    <script>
    document.addEventListener("DOMContentLoaded", function () {{
        const gd = document.querySelector('.plotly-graph-div');
        const toggleBtn = document.getElementById('rotate-toggle');
        const profileContent = document.getElementById('profile-content');
        const clusterContent = document.getElementById('cluster-content');

        const selectorSvg = document.getElementById('selector-svg');
        const toolDrawBtn = document.getElementById('tool-draw');
        const toolRotateBtn = document.getElementById('tool-rotate');
        const modeEllipseBtn = document.getElementById('mode-ellipse');
        const modeCircleBtn = document.getElementById('mode-circle');
        const clearSelectionBtn = document.getElementById('clear-selection');
        const resetSelectorViewBtn = document.getElementById('reset-selector-view');
        const selectorViewStatus = document.getElementById('selector-view-status');

        let autoRotate = true;
        let isDragging = false;
        const delta = {ROTATION_DELTA};

        let currentCamera = {{
            eye: {{ x: 1.8, y: 1.8, z: 0.9 }},
            up: {{ x: 0.0, y: 0.0, z: 1.0 }},
            center: {{ x: 0.0, y: 0.0, z: 0.0 }}
        }};

        let selectorTool = "draw";
        let selectorMode = "ellipse";

        let selectorView = {{
            yaw: 0.0,
            pitch: 0.0
        }};

        let selectorState = {{
            drawing: false,
            draggingShape: false,
            rotatingView: false,
            x0: null,
            y0: null,
            cx: null,
            cy: null,
            rx: null,
            ry: null,
            dragStartX: null,
            dragStartY: null,
            dragOrigCx: null,
            dragOrigCy: null,
            rotateStartX: null,
            rotateStartY: null,
            rotateOrigYaw: null,
            rotateOrigPitch: null,
            selectedIdx: []
        }};

        const coords = CLIENT_DATA.coords;
        const outcomes = CLIENT_DATA.outcomes;
        const studentIds = CLIENT_DATA.student_ids;
        const rawRecords = CLIENT_DATA.raw_records;
        const numericCols = CLIENT_DATA.numeric_cols;
        const categoricalCols = CLIENT_DATA.categorical_cols;
        const xScaled = CLIENT_DATA.x_scaled;
        const passCentroid = CLIENT_DATA.pass_centroid;
        const failCentroid = CLIENT_DATA.fail_centroid;
        const passLabel = CLIENT_DATA.pass_label;
        const failLabel = CLIENT_DATA.fail_label;
        const topK = CLIENT_DATA.top_k;
        const baseColors = CLIENT_DATA.base_colors;
        const nearBlackColor = CLIENT_DATA.near_black_color;

        // ------------------------------------------------------------
        // 3D camera behavior
        // ------------------------------------------------------------

        function cloneCamera(cam) {{
            return {{
                eye: {{ x: cam.eye.x, y: cam.eye.y, z: cam.eye.z }},
                up: {{ x: cam.up.x, y: cam.up.y, z: cam.up.z }},
                center: {{ x: cam.center.x, y: cam.center.y, z: cam.center.z }}
            }};
        }}

        function getSceneCamera() {{
            const layout = gd.layout || {{}};
            const scene = layout.scene || {{}};
            const cam = scene.camera || {{}};

            return {{
                eye: {{
                    x: (cam.eye && cam.eye.x != null) ? cam.eye.x : currentCamera.eye.x,
                    y: (cam.eye && cam.eye.y != null) ? cam.eye.y : currentCamera.eye.y,
                    z: (cam.eye && cam.eye.z != null) ? cam.eye.z : currentCamera.eye.z
                }},
                up: {{
                    x: (cam.up && cam.up.x != null) ? cam.up.x : currentCamera.up.x,
                    y: (cam.up && cam.up.y != null) ? cam.up.y : currentCamera.up.y,
                    z: (cam.up && cam.up.z != null) ? cam.up.z : currentCamera.up.z
                }},
                center: {{
                    x: (cam.center && cam.center.x != null) ? cam.center.x : currentCamera.center.x,
                    y: (cam.center && cam.center.y != null) ? cam.center.y : currentCamera.center.y,
                    z: (cam.center && cam.center.z != null) ? cam.center.z : currentCamera.center.z
                }}
            }};
        }}

        function rotateVectorAroundZ(v, theta) {{
            const c = Math.cos(theta);
            const s = Math.sin(theta);
            return {{
                x: c * v.x - s * v.y,
                y: s * v.x + c * v.y,
                z: v.z
            }};
        }}

        function stepRotation() {{
            const cam = getSceneCamera();
            const newEye = rotateVectorAroundZ(cam.eye, delta);

            currentCamera = {{
                eye: newEye,
                up: cam.up,
                center: cam.center
            }};

            Plotly.relayout(gd, {{
                "scene.camera": currentCamera
            }});
        }}

        function animate() {{
            if (autoRotate && !isDragging) {{
                stepRotation();
            }}
            requestAnimationFrame(animate);
        }}

        toggleBtn.addEventListener("click", function () {{
            autoRotate = !autoRotate;
            toggleBtn.textContent = autoRotate ? "Pause Rotation" : "Resume Rotation";
        }});

        gd.addEventListener("mousedown", function () {{
            isDragging = true;
        }});

        window.addEventListener("mouseup", function () {{
            if (isDragging) {{
                isDragging = false;
                currentCamera = cloneCamera(getSceneCamera());
            }}
        }});

        gd.addEventListener("touchstart", function () {{
            isDragging = true;
        }}, {{ passive: true }});

        window.addEventListener("touchend", function () {{
            if (isDragging) {{
                isDragging = false;
                currentCamera = cloneCamera(getSceneCamera());
            }}
        }}, {{ passive: true }});

        gd.on("plotly_relayout", function () {{
            currentCamera = cloneCamera(getSceneCamera());
        }});

        gd.on("plotly_hover", function (eventData) {{
            if (!eventData || !eventData.points || eventData.points.length === 0) return;
            const pt = eventData.points[0];
            const idx = Number(pt.customdata[0]);

            if (!Number.isNaN(idx) && idx >= 0 && idx < STUDENT_PROFILES.length) {{
                profileContent.innerHTML = STUDENT_PROFILES[idx];
            }}
        }});

        function resizePlot() {{
            Plotly.Plots.resize(gd);
        }}

        window.addEventListener("resize", resizePlot);
        setTimeout(resizePlot, 150);
        setTimeout(resizePlot, 500);

        // ------------------------------------------------------------
        // Selector geometry + rotatable projection
        // ------------------------------------------------------------

        const svgW = 380;
        const svgH = 250;
        const pad = 18;

        function clamp(val, lo, hi) {{
            return Math.min(Math.max(val, lo), hi);
        }}

        function svgToClientPoint(evt) {{
            const rect = selectorSvg.getBoundingClientRect();
            const clientX = evt.touches ? evt.touches[0].clientX : evt.clientX;
            const clientY = evt.touches ? evt.touches[0].clientY : evt.clientY;
            const x = ((clientX - rect.left) / rect.width) * svgW;
            const y = ((clientY - rect.top) / rect.height) * svgH;
            return {{ x, y }};
        }}

        function buildSvgElement(tag, attrs) {{
            const el = document.createElementNS("http://www.w3.org/2000/svg", tag);
            Object.entries(attrs).forEach(([k, v]) => el.setAttribute(k, v));
            return el;
        }}

        function rotatePointForSelector(p, yaw, pitch) {{
            const x = p[0];
            const y = p[1];
            const z = p[2];

            const cy = Math.cos(yaw);
            const sy = Math.sin(yaw);
            const cp = Math.cos(pitch);
            const sp = Math.sin(pitch);

            // yaw around z-axis
            const x1 = cy * x - sy * y;
            const y1 = sy * x + cy * y;
            const z1 = z;

            // pitch around x-axis
            const x2 = x1;
            const y2 = cp * y1 - sp * z1;
            const z2 = sp * y1 + cp * z1;

            return [x2, y2, z2];
        }}

        function getProjectedCoords() {{
            const rotated = coords.map(p => rotatePointForSelector(p, selectorView.yaw, selectorView.pitch));
            const xs = rotated.map(r => r[0]);
            const ys = rotated.map(r => r[1]);

            const xMin = Math.min(...xs);
            const xMax = Math.max(...xs);
            const yMin = Math.min(...ys);
            const yMax = Math.max(...ys);

            function xToSvg(x) {{
                const t = (x - xMin) / Math.max(1e-9, (xMax - xMin));
                return pad + t * (svgW - 2 * pad);
            }}

            function yToSvg(y) {{
                const t = (y - yMin) / Math.max(1e-9, (yMax - yMin));
                return svgH - (pad + t * (svgH - 2 * pad));
            }}

            return rotated.map(r => [xToSvg(r[0]), yToSvg(r[1]), r[2]]);
        }}

        function pointInsideSelector(svgX, svgY) {{
            if (
                selectorState.cx == null || selectorState.cy == null ||
                selectorState.rx == null || selectorState.ry == null ||
                selectorState.rx <= 1 || selectorState.ry <= 1
            ) {{
                return false;
            }}

            const dx = (svgX - selectorState.cx) / selectorState.rx;
            const dy = (svgY - selectorState.cy) / selectorState.ry;
            return (dx * dx + dy * dy) <= 1.0;
        }}

        function recomputeSelectedPoints() {{
            const projected = getProjectedCoords();
            const selected = [];
            for (let i = 0; i < projected.length; i++) {{
                const sx = projected[i][0];
                const sy = projected[i][1];
                if (pointInsideSelector(sx, sy)) {{
                    selected.push(i);
                }}
            }}
            selectorState.selectedIdx = selected;
        }}

        function updateSelectorStatus() {{
            const yawDeg = selectorView.yaw * 180 / Math.PI;
            const pitchDeg = selectorView.pitch * 180 / Math.PI;
            selectorViewStatus.textContent =
                `Selector view rotation — yaw: ${{yawDeg.toFixed(1)}}°, pitch: ${{pitchDeg.toFixed(1)}}°`;
        }}

        function renderSelector() {{
            selectorSvg.innerHTML = "";

            const bg = buildSvgElement("rect", {{
                x: 0, y: 0, width: svgW, height: svgH,
                fill: "rgba(255,255,255,0.02)"
            }});
            selectorSvg.appendChild(bg);

            const title = buildSvgElement("text", {{
                x: 12, y: 18,
                fill: "rgba(255,255,255,0.75)",
                "font-size": "11"
            }});
            title.textContent = "Rotatable selector projection";
            selectorSvg.appendChild(title);

            const projected = getProjectedCoords();
            const selectedSet = new Set(selectorState.selectedIdx);
            const hasSelection = selectorState.selectedIdx.length > 0;

            for (let i = 0; i < projected.length; i++) {{
                const cx = projected[i][0];
                const cy = projected[i][1];
                const isSelected = selectedSet.has(i);

                const point = buildSvgElement("circle", {{
                    cx: cx,
                    cy: cy,
                    r: isSelected ? 4.2 : 3.0,
                    fill: baseColors[i],
                    opacity: hasSelection ? (isSelected ? 1.0 : 0.82) : 0.90,
                    stroke: isSelected ? "white" : "none",
                    "stroke-width": isSelected ? 1.3 : 0
                }});
                selectorSvg.appendChild(point);
            }}

            if (
                selectorState.cx != null &&
                selectorState.cy != null &&
                selectorState.rx != null &&
                selectorState.ry != null &&
                selectorState.rx > 1 &&
                selectorState.ry > 1
            ) {{
                const shape = buildSvgElement("ellipse", {{
                    cx: selectorState.cx,
                    cy: selectorState.cy,
                    rx: selectorState.rx,
                    ry: selectorState.ry,
                    fill: "rgba(255,255,255,0.08)",
                    stroke: "rgba(255,255,255,0.95)",
                    "stroke-width": 1.6,
                    "stroke-dasharray": "5 4"
                }});
                selectorSvg.appendChild(shape);

                const centerHandle = buildSvgElement("circle", {{
                    cx: selectorState.cx,
                    cy: selectorState.cy,
                    r: 3.2,
                    fill: "rgba(255,255,255,0.95)"
                }});
                selectorSvg.appendChild(centerHandle);
            }}

            updateSelectorStatus();
        }}

        // ------------------------------------------------------------
        // Selector controls
        // ------------------------------------------------------------

        function setSelectorTool(tool) {{
            selectorTool = tool;
            toolDrawBtn.classList.toggle("active", tool === "draw");
            toolRotateBtn.classList.toggle("active", tool === "rotate");
            selectorSvg.style.cursor = tool === "rotate" ? "grab" : "crosshair";
        }}

        function setSelectorMode(mode) {{
            selectorMode = mode;
            modeEllipseBtn.classList.toggle("active", mode === "ellipse");
            modeCircleBtn.classList.toggle("active", mode === "circle");
        }}

        toolDrawBtn.addEventListener("click", function () {{
            setSelectorTool("draw");
        }});

        toolRotateBtn.addEventListener("click", function () {{
            setSelectorTool("rotate");
        }});

        modeEllipseBtn.addEventListener("click", function () {{
            setSelectorMode("ellipse");
        }});

        modeCircleBtn.addEventListener("click", function () {{
            setSelectorMode("circle");
        }});

        clearSelectionBtn.addEventListener("click", function () {{
            selectorState = {{
                drawing: false,
                draggingShape: false,
                rotatingView: false,
                x0: null,
                y0: null,
                cx: null,
                cy: null,
                rx: null,
                ry: null,
                dragStartX: null,
                dragStartY: null,
                dragOrigCx: null,
                dragOrigCy: null,
                rotateStartX: null,
                rotateStartY: null,
                rotateOrigYaw: null,
                rotateOrigPitch: null,
                selectedIdx: []
            }};
            renderSelector();
            update3DHighlight();
            updateClusterSummary();
        }});

        resetSelectorViewBtn.addEventListener("click", function () {{
            selectorView.yaw = 0.0;
            selectorView.pitch = 0.0;
            recomputeSelectedPoints();
            renderSelector();
            update3DHighlight();
            updateClusterSummary();
        }});

        function beginSelectorInteraction(evt) {{
            evt.preventDefault();
            const p = svgToClientPoint(evt);

            if (selectorTool === "rotate") {{
                selectorState.rotatingView = true;
                selectorState.rotateStartX = p.x;
                selectorState.rotateStartY = p.y;
                selectorState.rotateOrigYaw = selectorView.yaw;
                selectorState.rotateOrigPitch = selectorView.pitch;
                selectorSvg.style.cursor = "grabbing";
                return;
            }}

            if (
                selectorState.cx != null &&
                selectorState.cy != null &&
                selectorState.rx != null &&
                selectorState.ry != null &&
                pointInsideSelector(p.x, p.y)
            ) {{
                selectorState.draggingShape = true;
                selectorState.drawing = false;
                selectorState.dragStartX = p.x;
                selectorState.dragStartY = p.y;
                selectorState.dragOrigCx = selectorState.cx;
                selectorState.dragOrigCy = selectorState.cy;
                selectorSvg.style.cursor = "move";
                return;
            }}

            selectorState.drawing = true;
            selectorState.draggingShape = false;
            selectorState.x0 = p.x;
            selectorState.y0 = p.y;
            selectorState.cx = p.x;
            selectorState.cy = p.y;
            selectorState.rx = 1;
            selectorState.ry = 1;
            selectorState.selectedIdx = [];
            renderSelector();
        }}

        function moveSelectorInteraction(evt) {{
            if (!selectorState.drawing && !selectorState.draggingShape && !selectorState.rotatingView) return;
            evt.preventDefault();

            const p = svgToClientPoint(evt);

            if (selectorState.rotatingView) {{
                const dx = p.x - selectorState.rotateStartX;
                const dy = p.y - selectorState.rotateStartY;

                selectorView.yaw = selectorState.rotateOrigYaw + dx * 0.012;
                selectorView.pitch = clamp(selectorState.rotateOrigPitch - dy * 0.012, -1.45, 1.45);

                recomputeSelectedPoints();
                renderSelector();
                update3DHighlight();
                updateClusterSummary();
                return;
            }}

            if (selectorState.draggingShape) {{
                const dx = p.x - selectorState.dragStartX;
                const dy = p.y - selectorState.dragStartY;

                const minCx = selectorState.rx;
                const maxCx = svgW - selectorState.rx;
                const minCy = selectorState.ry;
                const maxCy = svgH - selectorState.ry;

                selectorState.cx = clamp(selectorState.dragOrigCx + dx, minCx, maxCx);
                selectorState.cy = clamp(selectorState.dragOrigCy + dy, minCy, maxCy);

                recomputeSelectedPoints();
                renderSelector();
                update3DHighlight();
                updateClusterSummary();
                return;
            }}

            const dx = p.x - selectorState.x0;
            const dy = p.y - selectorState.y0;

            selectorState.cx = (selectorState.x0 + p.x) / 2.0;
            selectorState.cy = (selectorState.y0 + p.y) / 2.0;

            let rx = Math.abs(dx) / 2.0;
            let ry = Math.abs(dy) / 2.0;

            if (selectorMode === "circle") {{
                const r = Math.max(rx, ry);
                rx = r;
                ry = r;
            }}

            selectorState.rx = Math.max(2, rx);
            selectorState.ry = Math.max(2, ry);

            recomputeSelectedPoints();
            renderSelector();
            update3DHighlight();
            updateClusterSummary();
        }}

        function endSelectorInteraction(evt) {{
            if (!selectorState.drawing && !selectorState.draggingShape && !selectorState.rotatingView) return;
            evt.preventDefault();

            selectorState.drawing = false;
            selectorState.draggingShape = false;
            selectorState.rotatingView = false;
            selectorSvg.style.cursor = selectorTool === "rotate" ? "grab" : "crosshair";

            recomputeSelectedPoints();
            renderSelector();
            update3DHighlight();
            updateClusterSummary();
        }}

        selectorSvg.addEventListener("mousedown", beginSelectorInteraction);
        selectorSvg.addEventListener("mousemove", moveSelectorInteraction);
        window.addEventListener("mouseup", endSelectorInteraction);

        selectorSvg.addEventListener("touchstart", beginSelectorInteraction, {{ passive: false }});
        selectorSvg.addEventListener("touchmove", moveSelectorInteraction, {{ passive: false }});
        window.addEventListener("touchend", endSelectorInteraction, {{ passive: false }});

        // ------------------------------------------------------------
        // 3D highlighting
        // ------------------------------------------------------------

        function update3DHighlight() {{
            const n = coords.length;
            const selSet = new Set(selectorState.selectedIdx);
            const hasSelection = selectorState.selectedIdx.length > 0;

            const colors = [];
            const sizes = [];

            for (let i = 0; i < n; i++) {{
                if (!hasSelection) {{
                    colors.push(baseColors[i]);
                    sizes.push({DEFAULT_MARKER_SIZE});
                }} else if (selSet.has(i)) {{
                    colors.push(baseColors[i]);
                    sizes.push({SELECTED_MARKER_SIZE});
                }} else {{
                    colors.push(nearBlackColor);
                    sizes.push({NONSELECTED_MARKER_SIZE});
                }}
            }}

            Plotly.restyle(gd, {{
                "marker.color": [colors],
                "marker.size": [sizes]
            }}, [0]);
        }}

        // ------------------------------------------------------------
        // Cluster summaries
        // ------------------------------------------------------------

        function mean(arr) {{
            if (!arr || arr.length === 0) return null;
            let s = 0;
            let c = 0;
            for (const v of arr) {{
                if (v != null && Number.isFinite(v)) {{
                    s += v;
                    c += 1;
                }}
            }}
            return c > 0 ? s / c : null;
        }}

        function stdAll(values) {{
            const m = mean(values);
            if (m == null) return 1.0;
            let s = 0;
            let c = 0;
            for (const v of values) {{
                if (v != null && Number.isFinite(v)) {{
                    s += (v - m) * (v - m);
                    c += 1;
                }}
            }}
            return c > 0 ? Math.sqrt(s / c) : 1.0;
        }}

        function euclid(a, b) {{
            let s = 0;
            for (let i = 0; i < a.length; i++) {{
                const d = a[i] - b[i];
                s += d * d;
            }}
            return Math.sqrt(s);
        }}

        function esc(str) {{
            return String(str)
                .replaceAll("&", "&amp;")
                .replaceAll("<", "&lt;")
                .replaceAll(">", "&gt;");
        }}

        function htmlList(items) {{
            if (!items || items.length === 0) return "<li>None identified</li>";
            return items.map(x => `<li>${{esc(x)}}</li>`).join("");
        }}

        function buildClusterSummary(selectedIdx) {{
            if (!selectedIdx || selectedIdx.length === 0) {{
                return `
                <div class="profile-title">No cluster selected yet</div>
                <p>
                    Draw a circle or ellipse in the lower-left selector. This panel will summarize the selected cluster,
                    its pass/fail composition, its centroid in embedding space, and its strongest distinguishing characteristics.
                </p>`;
            }}

            const selectedSet = new Set(selectedIdx);
            const otherIdx = [];
            for (let i = 0; i < coords.length; i++) {{
                if (!selectedSet.has(i)) otherIdx.push(i);
            }}

            const selectedOutcomes = selectedIdx.map(i => outcomes[i]);
            const outcomeCounts = {{}};
            selectedOutcomes.forEach(o => {{
                outcomeCounts[o] = (outcomeCounts[o] || 0) + 1;
            }});

            const selCoords = selectedIdx.map(i => coords[i]);
            const centroid3D = [0, 1, 2].map(j => mean(selCoords.map(r => r[j])));

            const selScaledCentroid = [];
            for (let j = 0; j < xScaled[0].length; j++) {{
                selScaledCentroid.push(mean(selectedIdx.map(i => xScaled[i][j])));
            }}

            const distPass = euclid(selScaledCentroid, passCentroid);
            const distFail = euclid(selScaledCentroid, failCentroid);
            const nearestGroup = distPass < distFail ? passLabel : failLabel;

            const numericSignals = [];
            for (const col of numericCols) {{
                const selVals = selectedIdx
                    .map(i => rawRecords[i][col])
                    .map(v => (v == null ? null : Number(v)))
                    .filter(v => Number.isFinite(v));

                const othVals = otherIdx
                    .map(i => rawRecords[i][col])
                    .map(v => (v == null ? null : Number(v)))
                    .filter(v => Number.isFinite(v));

                if (selVals.length === 0 || othVals.length === 0) continue;

                const selMean = mean(selVals);
                const othMean = mean(othVals);
                const globalStd = stdAll(
                    rawRecords.map(r => (r[col] == null ? null : Number(r[col]))).filter(v => Number.isFinite(v))
                ) || 1.0;

                const effect = Math.abs(selMean - othMean) / Math.max(1e-9, globalStd);

                const direction = selMean > othMean ? "higher" : "lower";
                numericSignals.push({{
                    score: effect,
                    text: `${{col}}: selected cluster mean is ${{direction}} (${{selMean.toFixed(3)}}) versus the rest of the map (${{othMean.toFixed(3)}})`
                }});
            }}

            numericSignals.sort((a, b) => b.score - a.score);

            const categoricalSignals = [];
            for (const col of categoricalCols) {{
                const selCounts = {{}};
                const othCounts = {{}};

                let selN = 0;
                let othN = 0;

                for (const i of selectedIdx) {{
                    const v = rawRecords[i][col];
                    if (v != null) {{
                        const key = String(v);
                        selCounts[key] = (selCounts[key] || 0) + 1;
                        selN += 1;
                    }}
                }}

                for (const i of otherIdx) {{
                    const v = rawRecords[i][col];
                    if (v != null) {{
                        const key = String(v);
                        othCounts[key] = (othCounts[key] || 0) + 1;
                        othN += 1;
                    }}
                }}

                if (selN === 0 || othN === 0) continue;

                for (const [cat, cnt] of Object.entries(selCounts)) {{
                    const pSel = cnt / selN;
                    const pOth = (othCounts[cat] || 0) / othN;
                    const diff = Math.abs(pSel - pOth);

                    categoricalSignals.push({{
                        score: diff,
                        text: `${{col}}='${{cat}}': ${{(100 * pSel).toFixed(1)}}% in selected cluster vs ${{(100 * pOth).toFixed(1)}}% outside`
                    }});
                }}
            }}

            categoricalSignals.sort((a, b) => b.score - a.score);

            const topSignals = [
                ...numericSignals.slice(0, Math.ceil(topK / 2)).map(x => x.text),
                ...categoricalSignals.slice(0, Math.floor(topK / 2)).map(x => x.text)
            ].slice(0, topK);

            const idsPreview = selectedIdx.slice(0, 12).map(i => studentIds[i]);
            const idsSuffix = selectedIdx.length > 12 ? ` ... and ${{selectedIdx.length - 12}} more` : "";

            const outcomeList = Object.entries(outcomeCounts)
                .sort((a, b) => b[1] - a[1])
                .map(([k, v]) => `${{k}}: ${{v}} (${{(100 * v / selectedIdx.length).toFixed(1)}}%)`);

            return `
                <div class="profile-card">
                    <div class="profile-title">Cluster Summary</div>
                    <p>
                        This selection contains <b>${{selectedIdx.length}}</b> students in the selector region you circled.
                    </p>

                    <div class="metric-grid">
                        <div class="metric-card">
                            <div class="label">Nearest group profile</div>
                            <div class="value">${{esc(nearestGroup)}}</div>
                        </div>
                        <div class="metric-card">
                            <div class="label">Outcome diversity</div>
                            <div class="value">${{Object.keys(outcomeCounts).length}}</div>
                        </div>
                        <div class="metric-card">
                            <div class="label">Distance to ${{esc(passLabel)}} centroid</div>
                            <div class="value">${{distPass.toFixed(3)}}</div>
                        </div>
                        <div class="metric-card">
                            <div class="label">Distance to ${{esc(failLabel)}} centroid</div>
                            <div class="value">${{distFail.toFixed(3)}}</div>
                        </div>
                    </div>

                    <hr class="panel-rule">

                    <div><b>Embedding centroid:</b> (${{centroid3D[0].toFixed(3)}}, ${{centroid3D[1].toFixed(3)}}, ${{centroid3D[2].toFixed(3)}})</div>

                    <div class="section-header">Outcome composition</div>
                    <ul>${{htmlList(outcomeList)}}</ul>

                    <div class="section-header">Strongest cluster-defining characteristics</div>
                    <ul>${{htmlList(topSignals)}}</ul>

                    <div class="section-header">Selected student IDs</div>
                    <p>${{idsPreview.map(esc).join(", ")}}${{idsSuffix}}</p>
                </div>
            `;
        }}

        function updateClusterSummary() {{
            clusterContent.innerHTML = buildClusterSummary(selectorState.selectedIdx);
        }}

        setSelectorTool("draw");
        setSelectorMode("ellipse");
        renderSelector();
        updateClusterSummary();
        update3DHighlight();
        animate();
    }});
    </script>
</body>
</html>
"""

output_file = Path(OUTPUT_HTML).resolve()
output_file.write_text(page_html, encoding="utf-8")
webbrowser.open(output_file.as_uri())

True